# ✂️ Bounding Memory: Trimming, Windowing and Running Summaries

## Learning Objectives
In this notebook, you will learn:
1. **Token-based trimming** - cut a history to a token budget with `trim_messages`
2. **Trimming inside a chain** - apply the trimmer to every call before the prompt
3. **Windowed histories** - store only the last *k* exchanges per session
4. **Running summaries** - fold older turns into a summary while recent turns stay verbatim

## Where this fits
`04_multi_user_sessions_and_persistence.ipynb` stored every message forever. Unbounded histories eventually
overflow the context window and make every call more expensive. This notebook applies the window and summary strategies of
`02_classic_memory_classes_and_chains.ipynb`, plus token trimming, with the `langchain_core` API.

## Prerequisites
- `02_classic_memory_classes_and_chains.ipynb` and `03_runnable_with_message_history.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (Part 1 makes no API calls)

> **⚠️ Legacy API**: Parts 3 and 4 use `RunnableWithMessageHistory` (deprecated in `langchain-core` 1.3.3) and a chat-message-history subclass (1.6.4), both removed in `langchain-core` 2.0; `trim_messages` (Parts 1–2) is current. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: answers the conversations and writes the summaries
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## ✂️ Part 1: Trimming a History by Tokens

`trim_messages` keeps as many messages as fit in `max_tokens`. It runs locally — the model is used only as a
token counter, no API call is made.

### Key Concepts:
- **`strategy="last"`**: keep the most recent messages that fit
- **`include_system=True`**: always keep the system message
- **`start_on="human"`**: the kept history starts with a human message, so it reads as a coherent conversation
- **`allow_partial=False`**: never cut a message in half

In [ ]:
# ============================================================================
# PART 1: WHICH MESSAGES SURVIVE A 45-TOKEN BUDGET?
# ============================================================================
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=llm,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

kept = trimmer.invoke(messages)
print(f"Original: {len(messages)} messages -> kept: {len(kept)}")
for m in kept:
    print(f"  {type(m).__name__:>13}: {m.content}")

Read the list above: the system message survives, the most recent turns survive, and the early turns — the
name "bob" and the ice cream — are gone. Everything that follows depends on which messages are in this list.

---
## 🔗 Part 2: Trimming Inside a Chain

`RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)` replaces the `messages` input with its
trimmed version before the prompt is built, on every call.

In [ ]:
# ============================================================================
# PART 2: A CHAIN THAT TRIMS BEFORE EVERY CALL
# ============================================================================
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in {language}. If the conversation does not contain the answer, say you don't know."),
    MessagesPlaceholder(variable_name="messages"),
])

trimmed_chain = RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer) | prompt | llm | StrOutputParser()

for question in ["What ice cream do I like?", "What math problem did I ask?"]:
    full = messages + [HumanMessage(content=question)]
    print(f"User: {question}")
    print(f"  (the model sees {len(trimmer.invoke(full))} of {len(full)} messages)")
    print(f"AI:   {trimmed_chain.invoke({'messages': full, 'language': 'English'})}\n")

Compare each answer with the "sees N of M messages" line: a question about something that was trimmed away
cannot be answered from the conversation, while a question about a kept turn can. Wrapping `trimmed_chain`
in `RunnableWithMessageHistory` (as in `03_runnable_with_message_history.ipynb`) stores the *full* history but
sends only the trimmed part to the model.

---
## 🪟 Part 3: A Windowed History (Keep the Last k Exchanges)

Instead of trimming on read, a history can refuse to grow: this subclass of `InMemoryChatMessageHistory`
drops the oldest messages whenever more than `k` exchanges are stored. Cost is fixed; older facts are lost.

In [ ]:
# ============================================================================
# PART 3A: A HISTORY THAT KEEPS ONLY k EXCHANGES
# ============================================================================
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


class WindowedChatHistory(InMemoryChatMessageHistory):
    """Chat history that keeps only the last k human/AI exchange pairs."""

    k: int = 2  # pydantic field

    def add_messages(self, messages) -> None:
        super().add_messages(messages)
        if len(self.messages) > self.k * 2:
            self.messages = self.messages[-(self.k * 2):]


windowed_store: dict[str, WindowedChatHistory] = {}


def get_windowed_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in windowed_store:
        windowed_store[session_id] = WindowedChatHistory(k=2)
    return windowed_store[session_id]


window_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Only use facts from this conversation."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
windowed_chain = RunnableWithMessageHistory(
    window_prompt | llm | StrOutputParser(), get_windowed_history,
    input_messages_key="input", history_messages_key="history",
)
print("✅ Windowed chain ready (k=2)")

In [ ]:
# ============================================================================
# PART 3B: WATCH THE WINDOW SLIDE
# ============================================================================
config = {"configurable": {"session_id": "windowed_test"}}

for text in ["My name is Paulo", "I live in Seattle", "I work as an AI engineer", "I have 2 cats",
             "What do you remember about me?"]:
    print(f"User: {text}")
    print(f"AI:   {windowed_chain.invoke({'input': text}, config=config)}")
    remembered = [m.content for m in windowed_store["windowed_test"].messages if isinstance(m, HumanMessage)]
    print(f"      [window holds {len(windowed_store['windowed_test'].messages)} messages: {remembered}]\n")

When the last question was asked, the window held only the two most recent exchanges — the job and the cats.
The name and the city had already slid out, so the answer can mention the job and the cats but not Paulo or
Seattle. Fixed memory means predictable cost and silently lost context.

---
## 🧾 Part 4: A Running Summary

The middle ground: keep the last `MAX_RECENT` messages verbatim and fold everything older into a running
summary written by a second, dedicated chain. Facts are compressed rather than deleted.

### Key Concepts:
- **Recent buffer**: the last `MAX_RECENT` messages, kept as-is
- **Running summary**: a short LLM-written summary of everything older, placed in the system prompt
- **Two chains**: one answers, one only summarises

In [ ]:
# ============================================================================
# PART 4A: THE ANSWER CHAIN AND THE SUMMARY CHAIN
# ============================================================================
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise.\n\nSummary of earlier conversation:\n{summary}"),
    MessagesPlaceholder(variable_name="recent_messages"),
    ("human", "{input}"),
])
chat_chain = chat_prompt | llm | StrOutputParser()

summarize_chain = ChatPromptTemplate.from_template(
    "Condense the current summary and new messages into a single updated summary (2-3 sentences). "
    "Preserve all key facts about the user.\n\n"
    "Current summary:\n{current_summary}\n\nNew messages:\n{new_messages}\n\nUpdated summary:"
) | llm | StrOutputParser()

MAX_RECENT = 4  # keep the last 2 exchanges verbatim
print("✅ Chains ready")

In [ ]:
# ============================================================================
# PART 4B: A CONVERSATION THAT OUTGROWS THE BUFFER
# ============================================================================
running_summary = ""
recent_messages = []

for user_input in ["My name is Paulo and I'm from Seattle",
                   "I work as an AI engineer building RAG systems",
                   "I have 2 cats named Luna and Milo",
                   "I'm building a LangChain course",
                   "What do you know about me? List everything."]:
    response = chat_chain.invoke({
        "summary": running_summary or "No prior conversation.",
        "recent_messages": recent_messages,
        "input": user_input,
    })
    print(f"User: {user_input}\nAI:   {response}")

    recent_messages += [HumanMessage(content=user_input), AIMessage(content=response)]
    if len(recent_messages) > MAX_RECENT:
        old, recent_messages = recent_messages[:-MAX_RECENT], recent_messages[-MAX_RECENT:]
        formatted = "\n".join(f"{'Human' if isinstance(m, HumanMessage) else 'AI'}: {m.content}" for m in old)
        running_summary = summarize_chain.invoke({
            "current_summary": running_summary or "None yet.",
            "new_messages": formatted,
        })
        print(f"  >>> summarised {len(old)} older messages -> {running_summary}")
    print()

print(f"Final summary: {running_summary}")
print(f"Recent messages kept verbatim: {len(recent_messages)}")

Unlike the window in Part 3, the final answer can draw on facts from the first turns (name, city, job) because
they survive in the summary, while the prompt stays bounded to one summary plus four recent messages. Check that
the list in the last answer includes them.

| Strategy | Keeps early facts | Extra LLM calls | Prompt size |
|---|---|---|---|
| Trim by tokens | ❌ | none | bounded by `max_tokens` |
| Window of k exchanges | ❌ | none | bounded by `k` |
| Running summary | ✅ (compressed) | one per overflow | bounded |

---
## 📝 Summary

In this notebook, we learned:

### 1. Token-based trimming
- `trim_messages` keeps the newest messages that fit a token budget, optionally pinning the system message

### 2. Trimming inside a chain
- `RunnablePassthrough.assign(messages=... | trimmer)` trims on every call; stored history stays complete

### 3. Windowed histories
- A history subclass that keeps `k` exchanges bounds storage and cost, but forgets early facts

### 4. Running summaries
- Older turns fold into a summary; recent turns stay verbatim; early facts survive in compressed form
- The same strategies on a LangGraph thread: [`02_memory_optimizations.ipynb`](06b_memory_optimizations.ipynb);
  agents built with `create_agent` get a ready-made `SummarizationMiddleware`

### Next Steps
- **`06_conversational_rag_with_history.ipynb`**: add conversation history to a retrieval-augmented QA chain